# 02 — Cleaning and preprocessing
Clean each dataset independently. Do not join the three datasets: they do not have verified common keys.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
RAW, PROCESSED = ROOT/"data/raw", ROOT/"data/processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

def normalize_missing(df):
    out = df.copy()
    for c in out.select_dtypes(include=["object", "string"]).columns:
        s = out[c].astype("string").str.strip()
        out[c] = s.replace({"": pd.NA, "NaN": pd.NA, "nan": pd.NA, "NULL": pd.NA,
                            "null": pd.NA, "None": pd.NA, "conditions NaN": pd.NA})
    return out

def numeric_from_text(series):
    return pd.to_numeric(series.astype("string").str.extract(r"(-?\d+(?:\.\d+)?)", expand=False), errors="coerce")

In [2]:
# A. Delivery dataset
delivery = normalize_missing(pd.read_csv(RAW/"delivery_time_train.csv", low_memory=False))
delivery["target_delivery_minutes"] = numeric_from_text(delivery["Time_taken(min)"])

numeric_cols = [
    "Delivery_person_Age", "Delivery_person_Ratings", "Restaurant_latitude",
    "Restaurant_longitude", "Delivery_location_latitude", "Delivery_location_longitude",
    "Vehicle_condition"
]
for c in numeric_cols:
    delivery[c] = pd.to_numeric(delivery[c], errors="coerce")

# Remove invalid/placeholder coordinates.
for c, lo, hi in [
    ("Restaurant_latitude", -90, 90), ("Delivery_location_latitude", -90, 90),
    ("Restaurant_longitude", -180, 180), ("Delivery_location_longitude", -180, 180)
]:
    delivery.loc[(delivery[c] < lo) | (delivery[c] > hi) | (delivery[c] == 0), c] = np.nan

before = len(delivery)
delivery = delivery.drop_duplicates()
delivery = delivery[delivery["target_delivery_minutes"].between(1, 180)]
print("Delivery rows:", before, "->", len(delivery))
display(delivery.head())
delivery.to_csv(PROCESSED/"delivery_train_clean.csv", index=False)

Delivery rows: 45593 -> 45593


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),target_delivery_minutes
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,...,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24,24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,...,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33,33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,...,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26,26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,...,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21,21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,...,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30,30


In [3]:
# B. Order-level data
orders_path = RAW/"online_food_orders.csv"
if orders_path.exists():
    orders = normalize_missing(pd.read_csv(orders_path, low_memory=False))
    for c in ["cost_of_the_order", "food_preparation_time", "delivery_time"]:
        if c in orders:
            orders[c] = pd.to_numeric(orders[c], errors="coerce")
    if "rating" in orders:
        orders["rating_numeric"] = pd.to_numeric(
            orders["rating"].replace({"Not given": pd.NA}), errors="coerce"
        )
    before = len(orders)
    orders = orders.drop_duplicates()
    print("Online order rows:", before, "->", len(orders))
    orders.to_csv(PROCESSED/"online_orders_clean.csv", index=False)
    display(orders.head())
else:
    print("Online order dataset not found.")

Online order rows: 1898 -> 1898


,order_id,customer_id,restaurant_name,cuisine_type,cost_of_the_order,day_of_the_week,rating,food_preparation_time,delivery_time,rating_numeric
0,1477147,337525,Hangawi,Korean,30.75,Weekend,Not given,25,20,<NA>
1,1477685,358141,Blue Ribbon Sushi Izakaya,Japanese,12.08,Weekend,Not given,25,23,<NA>
2,1477070,66393,Cafe Habana,Mexican,12.23,Weekday,5,23,28,5
3,1477334,106968,Blue Ribbon Fried Chicken,American,29.20,Weekend,3,25,15,3
4,1478249,76942,Dirty Bird to Go,American,11.59,Weekday,4,25,24,4


In [4]:
# C. Restaurant listings
listings_path = RAW/"swiggy_restaurant_listings.csv"
if listings_path.exists():
    raw_listings = pd.read_csv(listings_path, encoding="latin1", low_memory=False)
    raw_listings.columns = raw_listings.columns.astype(str).str.strip()

    def first_number(v):
        if pd.isna(v): return np.nan
        m = re.search(r"\d+(?:\.\d+)?", str(v).replace(",", ""))
        return float(m.group()) if m else np.nan

    def count_number(v):
        if pd.isna(v): return 0.0
        s = str(v).lower().replace(",", "")
        m = re.search(r"(\d+(?:\.\d+)?)\s*([km]?)", s)
        if not m: return 0.0
        return float(m.group(1)) * {"": 1, "k": 1000, "m": 1_000_000}[m.group(2)]

    listings = pd.DataFrame({
        "restaurant_name": raw_listings["Restaurant Name"].fillna("").astype(str).str.strip(),
        "cuisine": raw_listings["Cuisine"].fillna("").astype(str).str.strip(),
        "rating": raw_listings["Rating"].map(first_number),
        "rating_count": raw_listings["Number of Ratings"].map(count_number),
        "average_price": raw_listings["Average Price"].map(first_number),
        "number_of_offers": pd.to_numeric(raw_listings["Number of Offers"], errors="coerce"),
        "offer_name": raw_listings["Offer Name"].fillna("").astype(str).str.strip(),
        "area": raw_listings["Area"].fillna("").astype(str).str.strip(),
        "pure_veg": raw_listings["Pure Veg"].fillna("").astype(str).str.strip(),
        "location": raw_listings["Location"].fillna("").astype(str).str.strip(),
    })
    listings = listings[listings["restaurant_name"].ne("")]
    listings = listings.drop_duplicates(["restaurant_name", "area", "location"]).reset_index(drop=True)
    listings.to_csv(PROCESSED/"restaurants_clean.csv", index=False)
    print("Restaurant rows:", len(raw_listings), "->", len(listings))
    display(listings.head())
else:
    print("Restaurant listings not found.")

Restaurant rows: 140657 -> 139314


,restaurant_name,cuisine,rating,rating_count,average_price,number_of_offers,offer_name,area,pure_veg,location
0,La Pino'Z Pizza,"Pizzas, Pastas",4.0,10.0,250.0,2,FLAT DEAL\nFLAT â¹125 OFF\nUSE FLAT125ABOVE â...,LALA LAJPAT RAI MARKET,No,Abohar
1,The Second Wife,"Indian, North Indian",3.6,50.0,250.0,2,"30% OFF UPTO â¹75\nUSE TRYNEWABOVE â¹149, FL...",Central Abohar,No,Abohar
2,Tasty Bites,"Italian, Beverages",3.8,10.0,200.0,1,FLAT â¹120 OFF\nUSE AXIS120ABOVE â¹500,Central Abohar,Yes,Abohar
3,Food Studio,"Pizzas, Burgers",3.5,8.0,49.0,5,"50% OFF UPTO â¹100\nUSE TRYNEWABOVE â¹129, F...",Central Abohar,Yes,Abohar
4,Roll Express,"Fast Food, Snacks",4.3,100.0,200.0,2,DEAL OF DAY\n10% OFF UPTO â¹40\nUSE STEALDEAL...,Circular Road,No,Abohar
